# 03 — Model Comparison: XGBoost vs MLP vs Meta CHM

This notebook presents the main results of the canopy height model comparison study. We evaluate three approaches against ALS LiDAR ground truth from the EBA campaign.

## Models Compared

| Model | Input | Architecture | Training |
|-------|-------|-------------|----------|
| **Meta Global CHM** | High-res satellite imagery | Vision Transformer + CNN decoder | Global (no local training) |
| **XGBoost** | Google AlphaEarth embeddings (64-dim) | Gradient-boosted trees (500 trees, depth 6) | Local (70/15/15 spatial split) |
| **Fine-tuned MLP** | Google AlphaEarth embeddings (64-dim) | 4-layer residual MLP (256 hidden, LayerNorm) | Local (same split as XGBoost) |

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from src.config import DATA_DIR, MODEL_DIR

# Load metrics
metrics_path = DATA_DIR / "sample" / "three_way_comparison.csv"
if metrics_path.exists():
    metrics_df = pd.read_csv(metrics_path)
    print("Model comparison metrics:")
    print(metrics_df.to_string(index=False))
else:
    print(f"Metrics file not found at {metrics_path}")
    print("Run `make compare` to generate it.")
    metrics_df = None

## Results Summary

### Test set performance (30m resolution, spatially independent transects)

| Model | MAE (m) | RMSE (m) | R² | Bias (m) |
|-------|---------|----------|-----|----------|
| Meta Global CHM | 4.98 | 7.16 | 0.287 | -1.47 |
| XGBoost | 3.62 | 5.28 | 0.612 | +0.01 |
| Fine-tuned MLP | 3.56 | 5.27 | 0.613 | +0.31 |

The embedding-based models reduce MAE by ~29% compared to Meta's global product, with R² more than doubling (0.29 → 0.61).

In [ ]:
# Display the comparison figures
from IPython.display import Image, display

figures_dir = Path("..") / "figures"

for fig_name, title in [
    ("three_way_comparison.png", "Scatter plots: all models vs ALS CHM"),
    ("three_way_metrics_bars.png", "Bar chart: key metrics"),
    ("three_way_rasters.png", "Wall-to-wall predictions over AOI"),
]:
    fig_path = figures_dir / fig_name
    if fig_path.exists():
        print(f"\n{'='*60}")
        print(title)
        print(f"{'='*60}")
        display(Image(filename=str(fig_path), width=900))
    else:
        print(f"Figure not found: {fig_path}")

## Analysis: Why does XGBoost match the MLP?

The near-identical performance of XGBoost and the MLP (MAE difference of only 0.06 m) is noteworthy. Several factors explain this:

1. **Pre-computed embeddings do the heavy lifting** — Google's foundation model has already learned rich feature representations from satellite imagery. The downstream task (regression from 64 features to 1 target) is relatively simple.

2. **XGBoost excels at tabular regression** — for structured features of moderate dimensionality, gradient-boosted trees are hard to beat. The MLP's capacity for learning complex interactions isn't needed when the feature space is already well-structured.

3. **Limited training data** — with ~1.5M samples from 487 transects, there may not be enough data diversity for the MLP to find patterns that XGBoost misses.

## Meta's systematic underestimation

The -1.47 m bias in Meta's product reveals systematic underestimation of tall canopy in this region. Possible causes:

- **Training data distribution** — Meta's model was trained globally; the cerrado-Amazon transition may be underrepresented
- **Temporal mismatch** — Meta's product uses recent imagery while ALS data is from 2016–2018
- **Mixed pixels** — in heterogeneous landscapes, 1m pixels at forest edges may average canopy and gaps differently than LiDAR

## Individual Model Results

### XGBoost (train/val/test scatter plots)

In [ ]:
fig_path = figures_dir / "xgboost_results.png"
if fig_path.exists():
    display(Image(filename=str(fig_path), width=900))

### Fine-tuned MLP (train/val/test scatter plots)

In [ ]:
fig_path = figures_dir / "prithvi_results.png"
if fig_path.exists():
    display(Image(filename=str(fig_path), width=900))

## Conclusions

1. **Local fine-tuning with foundation model embeddings substantially outperforms global products** — a 29% MAE reduction with minimal effort.

2. **XGBoost is the recommended model** for this use case — it matches MLP accuracy while being faster to train, easier to deploy, and more interpretable.

3. **Google AlphaEarth embeddings are a powerful, zero-effort feature set** for vegetation structure prediction — no feature engineering, band selection, or index computation required.

4. **Spatial cross-validation is essential** — without it, spatial autocorrelation would inflate R² and understate prediction error for new locations.